In [ ]:
# SARIMA Model Training - AgriPrice AI

# This notebook trains and evaluates the SARIMA (Seasonal AutoRegressive Integrated Moving Average) model for commodity price forecasting.

## Steps:
# 1. **Load Processed Data** - Prepared dataset with features
# 2. **Stationarity Check** - Augmented Dickey-Fuller test
# 3. **Model Training** - SARIMA(1,1,1)(1,1,1,12) with validation
# 4. **Forecasting** - 30-day price predictions
# 5. **Evaluation** - Performance metrics (MAE, RMSE, MAPE)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load processed data
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir)
processed_data_path = os.path.join(base_dir, "data", "processed_data.csv")
models_dir = os.path.join(base_dir, "models")
os.makedirs(models_dir, exist_ok=True)

data = pd.read_csv(processed_data_path, parse_dates=["Date"], index_col="Date").sort_index()

print(f"✅ Loaded processed data: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")
print(f"\nPrice statistics:")
print(f"  Mean: Rs.{data['Price'].mean():.2f}")
print(f"  Std: Rs.{data['Price'].std():.2f}")

In [ ]:
## Step 1: Stationarity Check (Augmented Dickey-Fuller Test)
print("\n" + "=" * 60)
print("STATIONARITY CHECK (ADF TEST)")
print("=" * 60)

adf_result = adfuller(data["Price"].dropna())
adf_statistic, adf_pvalue, adf_lags, adf_obs = adf_result[0], adf_result[1], adf_result[2], adf_result[3]

print(f"\nAugmented Dickey-Fuller Test Results:")
print(f"  Test Statistic: {adf_statistic:.6f}")
print(f"  P-Value: {adf_pvalue:.6f}")
print(f"  Number of Lags: {adf_lags}")
print(f"  Observations: {adf_obs}")

if adf_pvalue < 0.05:
    print(f"\n✅ Series is STATIONARY (p={adf_pvalue:.4f} < 0.05)")
    print("   → Ready for ARIMA modeling without differencing")
else:
    print(f"\n⚠️  Series is NON-STATIONARY (p={adf_pvalue:.4f} >= 0.05)")
    print("   → Differencing will be applied by SARIMA (d=1)")

In [ ]:
## Step 2: Train SARIMA Model
print("\n" + "=" * 60)
print("SARIMA MODEL TRAINING")
print("=" * 60)
print("Training SARIMA(1,1,1)(1,1,1,12) model...")
print("Parameters:")
print("  p=1, d=1, q=1 (AR, Differencing, MA)")
print("  P=1, D=1, Q=1, s=12 (Seasonal: yearly cycle)")

try:
    sarima_model = SARIMAX(
        data["Price"],
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    sarima_results = sarima_model.fit(disp=False)
    print("\n✅ Model trained successfully!")
    print(f"\nModel Summary:")
    print(sarima_results.summary())
except Exception as e:
    print(f"\n❌ Training error: {e}")
    raise

In [ ]:
## Step 3: Model Evaluation & Backtest
print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

# In-sample predictions (backtest)
test_len = min(30, max(1, len(data) // 3))
start_idx = len(data) - test_len
predictions = sarima_results.predict(start=start_idx, end=len(data) - 1)
actual = data["Price"].iloc[start_idx:]

# Trim predictions to match actual length if needed
predictions = predictions[: len(actual)]

mae = mean_absolute_error(actual, predictions)
rmse = np.sqrt(mean_squared_error(actual, predictions))
mape = np.nanmean(np.abs((actual.values - np.array(predictions)) / actual.values)) * 100
accuracy = 100 - mape

print(f"\nBacktest Results ({len(actual)} observations):")
print(f"  MAE (Mean Absolute Error):  Rs.{mae:.2f}")
print(f"  RMSE (Root Mean Sq Error):  Rs.{rmse:.2f}")
print(f"  MAPE (Mean Absolute % Error): {mape:.2f}%")
print(f"  ✓ Accuracy: {accuracy:.2f}%")

# Plot actual vs predicted
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(actual.index, actual.values, label="Actual", color="blue", linewidth=2, marker="o")
ax.plot(actual.index, predictions.values, label="Predicted", color="red", linestyle="--", linewidth=2, marker="s")
ax.set_title("In-Sample Predictions vs Actual Prices", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Price (Rs/Quintal)")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## Step 4: 30-Day Forecast
print("\n" + "=" * 60)
print("30-DAY PRICE FORECAST")
print("=" * 60)

forecast_steps = 30
forecast_result = sarima_results.get_forecast(steps=forecast_steps)
forecast = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int()

# Create forecast dataframe
forecast_dates = pd.date_range(start=data.index[-1] + pd.Timedelta(days=1), periods=forecast_steps)
forecast_df = pd.DataFrame({
    "Date": forecast_dates,
    "Forecast": forecast.values,
    "Lower_CI": forecast_ci.iloc[:, 0].values,
    "Upper_CI": forecast_ci.iloc[:, 1].values,
})

print(f"\nNext 30-Day Price Forecast:")
print(forecast_df.to_string(index=False))

# Plot historical + forecast
fig, ax = plt.subplots(figsize=(14, 6))
history = data["Price"].iloc[-90:]  # Last 90 days
ax.plot(history.index, history.values, label="Historical (90d)", color="blue", linewidth=2)
ax.plot(forecast_df["Date"], forecast_df["Forecast"], label="30-Day Forecast", color="red", linestyle="--", linewidth=2, marker="o")
ax.fill_between(forecast_df["Date"], forecast_df["Lower_CI"], forecast_df["Upper_CI"], alpha=0.2, color="red", label="95% Confidence Interval")
ax.set_title("Historical Prices & 30-Day Forecast", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Price (Rs/Quintal)")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print key forecast statistics
print(f"\nForecast Statistics:")
print(f"  Current Price: Rs.{data['Price'].iloc[-1]:.2f}")
print(f"  Min Forecast: Rs.{forecast.min():.2f}")
print(f"  Max Forecast: Rs.{forecast.max():.2f}")
print(f"  Avg Forecast: Rs.{forecast.mean():.2f}")
change_pct = ((forecast.iloc[-1] - data['Price'].iloc[-1]) / data['Price'].iloc[-1]) * 100
print(f"  Expected Change (30d): {change_pct:+.2f}%")

if change_pct > 20:
    print(f"\n🚨 ALERT: Price spike predicted (+{change_pct:.1f}%)")
    print("   → Buffer stock release may be recommended")

In [ ]:
## Step 5: Save Model
print("\n" + "=" * 60)
print("SAVING MODEL")
print("=" * 60)

model_path = os.path.join(models_dir, "sarima_model.pkl")
joblib.dump(sarima_results, model_path)
print(f"✅ Model saved to: {model_path}")

# Save forecast for reference
forecast_path = os.path.join(models_dir, "forecast_30days.csv")
forecast_df.to_csv(forecast_path, index=False)
print(f"✅ Forecast saved to: {forecast_path}")